In [ ]:
import os

print(os.listdir("/content"))

['.config', '.ipynb_checkpoints', 'data.ml', 'data.en', 'sample_data']


In [ ]:
with open("/content/data.ml", "r", encoding="utf-8") as f:
    malayalam = f.readlines()

with open("/content/data.en", "r", encoding="utf-8") as f:
    english = f.readlines()

print("Malayalam sentences:", len(malayalam))
print("English sentences:", len(english))

print("\nMalayalam example:", malayalam[0].strip())
print("English example:", english[0].strip())

Malayalam sentences: 176731
English sentences: 176730

Malayalam example: ഇപ്പോൾ വീടുകളിലുണ്ടാക്കുന്ന രാഖികൾ ഇഷ്ടപ്പെടുന്നവരാണേറെയും.
English example: People prefer homemade Rakhis these days.


In [ ]:
import pandas as pd
import re

# Handle alignment mismatch by truncating to the minimum length
min_len = min(len(english), len(malayalam))
if len(english) != len(malayalam):
    print(f"Warning: Mismatch in line counts! Malayalam has {len(malayalam)} lines, English has {len(english)} lines.")
    print(f"Truncating both to the minimum length of {min_len} to maintain alignment.")

# Create a paired dataset up to min_len
df = pd.DataFrame({
    "malayalam": [s.strip() for s in malayalam[:min_len]],
    "english": [s.strip() for s in english[:min_len]]
})

print("Original sentence pairs:", len(df))

# Remove empty sentences
df = df[
    (df["malayalam"] != "") &
    (df["english"] != "")
].copy()

print("After removing empty pairs:", len(df))

# Remove exact duplicate pairs
df = df.drop_duplicates(
    subset=["malayalam", "english"]
).reset_index(drop=True)

print("After removing duplicates:", len(df))

# Normalize repeated whitespace
for col in ["malayalam", "english"]:
    df[col] = df[col].apply(
        lambda x: re.sub(r"\s+", " ", x).strip()
    )

# Filter sentence lengths using word counts
df["ml_words"] = df["malayalam"].str.split().str.len()
df["en_words"] = df["english"].str.split().str.len()

df = df[
    (df["ml_words"] >= 2) &
    (df["en_words"] >= 2) &
    (df["ml_words"] <= 50) &
    (df["en_words"] <= 50)
].copy()

# Remove temporary length columns
df = df.drop(columns=["ml_words", "en_words"])
df = df.reset_index(drop=True)

print("Final cleaned sentence pairs:", len(df))
print("\nSample cleaned data:")
print(df.head(10).to_string(index=False))

Truncating both to the minimum length of 176730 to maintain alignment.
Original sentence pairs: 176730
After removing empty pairs: 176726
After removing duplicates: 151280
Final cleaned sentence pairs: 145021

Sample cleaned data:
                                                                                           malayalam                                                                                                 english
                                         ഇപ്പോൾ വീടുകളിലുണ്ടാക്കുന്ന രാഖികൾ ഇഷ്ടപ്പെടുന്നവരാണേറെയും.                                                               People prefer homemade Rakhis these days.
                                        ജലീലും മറ്റ് വിശിഷ്ട വ്യക്തികളും ചടങ്ങിൽ സന്നിഹിതരായിരുന്നു.                                         Jaleel and other dignitaries were also present on the occasion.
ഈ രണ്ട് ഇടനാഴികളും നിര്‍മ്മാണം പൂര്‍ത്തിയാകുമ്പോള്‍ ഈ പ്രദേശത്തെ പൊതു ഗതാഗതത്തെ ഗണ്യമായി സഹായിക്കും.                The two corridors, once completed, wil

In [ ]:

from sklearn.model_selection import train_test_split

# 80% training, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Split remaining 20% equally:
# 10% validation, 10% testing
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Training pairs:", len(train_df))
print("Validation pairs:", len(val_df))
print("Testing pairs:", len(test_df))


Training pairs: 116016
Validation pairs: 14502
Testing pairs: 14503


In [ ]:

df.to_csv(
    "/content/cleaned_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

train_df.to_csv(
    "/content/train.csv",
    index=False,
    encoding="utf-8-sig"
)

val_df.to_csv(
    "/content/validation.csv",
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    "/content/test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("All dataset files saved successfully!")


All dataset files saved successfully!


In [ ]:
from google.colab import files

files.download("/content/cleaned_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Dataset shape:", df.shape)
print("Missing values:\n", df.isnull().sum())

print("\nFinal samples:")
display(df.sample(min(5, len(df)), random_state=42))

Dataset shape: (145021, 2)
Missing values:
 malayalam    0
english      0
dtype: int64

Final samples:


,malayalam,english
139087,സഹകരണ ഉല്‍പ്പന്നങ്ങളുടെ ബ്രാന്‍ഡിംഗും വിപണനവും...,Branding and marketing of cooperative products...
5601,"അവരുടെ സാമൂഹിക, സാമ്പത്തിക, വിദ്യാഭ്യാസപരമായ ശ...",The Bill provides a mechanism for their social...
198,"ഞങ്ങൾ അത് ആത്മാർത്ഥമായാണ് ചെയ്യുന്നത്, എന്തുകൊ...",We are doing it sincerely because we value Ger...
61618,10 പൈതൃക കോളേജുകള്‍ക്ക്‌ 10 കോടി രൂപയും അനുവദി...,10 crores of rupees also is allocated for ten ...
107431,ഒരു വ്യക്തിഗത കെട്ടിടത്തിന്റെ ഉള്ളിലേക്കുള്ള പ...,Provided also that internal access to an indiv...


In [ ]:
import pandas as pd

df = pd.read_csv("/content/cleaned_dataset.csv")

print("Dataset size:", df.shape)
print("Column names:", df.columns.tolist())
print("Missing values:\n", df.isnull().sum())

display(df.head(10))

Dataset size: (145021, 2)
Column names: ['malayalam', 'english']
Missing values:
 malayalam    0
english      0
dtype: int64


,malayalam,english
0,ഇപ്പോൾ വീടുകളിലുണ്ടാക്കുന്ന രാഖികൾ ഇഷ്ടപ്പെടുന...,People prefer homemade Rakhis these days.
1,ജലീലും മറ്റ് വിശിഷ്ട വ്യക്തികളും ചടങ്ങിൽ സന്നി...,Jaleel and other dignitaries were also present...
2,ഈ രണ്ട് ഇടനാഴികളും നിര്‍മ്മാണം പൂര്‍ത്തിയാകുമ്...,"The two corridors, once completed, will greatl..."
3,മെഡിക്കൽ യുജി സീറ്റുകളിൽ കഴിഞ്ഞ നാലരവർഷം കൊണ്ട...,The number of UG medical seats have been incre...
4,ചമ്പാരൻ സത്യാഗ്രഹത്തിന്റെ നൂറാം വാര്‍ഷികത്തോടന...,Three Heritage Books on the occasion of Centen...
5,റഷ്യയുമായുള്ള ബന്ധം ശക്തമാക്കുന്നു.,Strengthening the ties with Russia.
6,ഇന്ത്യൻ ഡെന്റൽ അസോസിയേഷൻ സെക്രട്ടറി ജനറൽ ഡോ അശ...,"Dr Ashok Dhoble , Secretary General, Indian de..."
7,2009 ലെ 17ൽ നിന്നും വിദ്യാര്‍ത്ഥി പ്രവേശനം 201...,The students enrolment went up from 17 in 2009...
8,"നമ്മുടെ യുവാക്കളുടെ നൂതനാശയങ്ങളിലും, കഠിന പ്രയ...",The Government is proud of the hard work and i...
9,ഇലക്ട്രിക് വാഹനങ്ങളുടെ ചില ഭാഗങ്ങളെ കസ്റ്റംസ് ...,Customs duty exempted on certain parts of elec...


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42
)

train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/validation.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 116016
Validation: 14502
Testing: 14503


In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (116016, 2)
Validation: (14502, 2)
Testing: (14503, 2)


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Separate source and target text
train_ml = train_df["malayalam"].astype(str).tolist()
train_en = train_df["english"].astype(str).tolist()

val_ml = val_df["malayalam"].astype(str).tolist()
val_en = val_df["english"].astype(str).tolist()

test_ml = test_df["malayalam"].astype(str).tolist()
test_en = test_df["english"].astype(str).tolist()

# Create separate tokenizers
ml_tokenizer = Tokenizer(oov_token="<OOV>")
en_tokenizer = Tokenizer(
    oov_token="<OOV>",
    filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n'
)

# Fit only on training data
ml_tokenizer.fit_on_texts(train_ml)
en_tokenizer.fit_on_texts(train_en)

print("Malayalam vocabulary:", len(ml_tokenizer.word_index) + 1)
print("English vocabulary:", len(en_tokenizer.word_index) + 1)

Malayalam vocabulary: 179055
English vocabulary: 43016


In [ ]:
# Convert sentences into integer sequences
X_train = ml_tokenizer.texts_to_sequences(train_ml)
Y_train = en_tokenizer.texts_to_sequences(train_en)

X_val = ml_tokenizer.texts_to_sequences(val_ml)
Y_val = en_tokenizer.texts_to_sequences(val_en)

X_test = ml_tokenizer.texts_to_sequences(test_ml)
Y_test = en_tokenizer.texts_to_sequences(test_en)

print("Example Malayalam sequence:", X_train[0])
print("Example English sequence:", Y_train[0])

Example Malayalam sequence: [4682, 7182, 46, 16, 26, 18908, 4682, 389, 73299, 1740, 73300, 17006, 813, 1376]
Example English sequence: [2729, 2866, 304, 1875, 25445, 3, 2729, 144, 3787, 3, 39, 2, 1462, 247, 5, 17, 31, 35, 189, 12, 5403, 575]


In [ ]:
import numpy as np

# Estimate suitable lengths from the training data
MAX_ML_LEN = int(np.percentile(
    [len(s) for s in X_train], 95
))
MAX_EN_LEN = int(np.percentile(
    [len(s) for s in Y_train], 95
))

MAX_ML_LEN = max(1, MAX_ML_LEN)
MAX_EN_LEN = max(1, MAX_EN_LEN)

X_train_pad = pad_sequences(
    X_train, maxlen=MAX_ML_LEN,
    padding="post", truncating="post"
)
Y_train_pad = pad_sequences(
    Y_train, maxlen=MAX_EN_LEN,
    padding="post", truncating="post"
)

X_val_pad = pad_sequences(
    X_val, maxlen=MAX_ML_LEN,
    padding="post", truncating="post"
)
Y_val_pad = pad_sequences(
    Y_val, maxlen=MAX_EN_LEN,
    padding="post", truncating="post"
)

X_test_pad = pad_sequences(
    X_test, maxlen=MAX_ML_LEN,
    padding="post", truncating="post"
)
Y_test_pad = pad_sequences(
    Y_test, maxlen=MAX_EN_LEN,
    padding="post", truncating="post"
)

print("Input training shape:", X_train_pad.shape)
print("Target training shape:", Y_train_pad.shape)
print("Maximum Malayalam length:", MAX_ML_LEN)
print("Maximum English length:", MAX_EN_LEN)

Input training shape: (116016, 26)
Target training shape: (116016, 38)
Maximum Malayalam length: 26
Maximum English length: 38


In [ ]:

from tensorflow.keras.preprocessing.text import Tokenizer

MAX_ML_VOCAB = 30000
MAX_EN_VOCAB = 20000

ml_tokenizer = Tokenizer(
    num_words=MAX_ML_VOCAB,
    oov_token="<OOV>"
)

en_tokenizer = Tokenizer(
    num_words=MAX_EN_VOCAB,
    oov_token="<OOV>"
)

# Learn vocabularies from training data only
ml_tokenizer.fit_on_texts(
    train_df["malayalam"].astype(str)
)

en_tokenizer.fit_on_texts(
    train_df["english"].astype(str)
)

print("Malayalam vocabulary limit:", MAX_ML_VOCAB)
print("English vocabulary limit:", MAX_EN_VOCAB)

print("Malayalam words observed:",
      len(ml_tokenizer.word_index))

print("English words observed:",
      len(en_tokenizer.word_index))


Malayalam vocabulary limit: 30000
English vocabulary limit: 20000
Malayalam words observed: 179054
English words observed: 43009


In [ ]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

def make_sequences(df, column, tokenizer):
    return tokenizer.texts_to_sequences(
        df[column].astype(str).tolist()
    )

X_train = make_sequences(train_df, "malayalam", ml_tokenizer)
Y_train = make_sequences(train_df, "english", en_tokenizer)

X_val = make_sequences(val_df, "malayalam", ml_tokenizer)
Y_val = make_sequences(val_df, "english", en_tokenizer)

X_test = make_sequences(test_df, "malayalam", ml_tokenizer)
Y_test = make_sequences(test_df, "english", en_tokenizer)

print("Malayalam example:", X_train[0])
print("English example:", Y_train[0])


Malayalam example: [4682, 7182, 46, 16, 26, 18908, 4682, 389, 1, 1740, 1, 17006, 813, 1376]
English example: [2729, 2866, 304, 1875, 1, 3, 2729, 144, 3787, 3, 39, 2, 1462, 247, 5, 17, 31, 35, 189, 12, 5403, 575]


In [ ]:

def oov_rate(sequences):
    total = sum(len(s) for s in sequences)
    unknown = sum(
        token == 1
        for s in sequences
        for token in s
    )
    return 100 * unknown / total if total else 0

print("Malayalam OOV rate:",
      round(oov_rate(X_train), 2), "%")

print("English OOV rate:",
      round(oov_rate(Y_train), 2), "%")


Malayalam OOV rate: 15.98 %
English OOV rate: 1.44 %


In [ ]:

# Increase Malayalam vocabulary limit
ml_tokenizer.num_words = 50000

# Keep English vocabulary limit unchanged
en_tokenizer.num_words = 20000

# Regenerate sequences using the updated limits
X_train = ml_tokenizer.texts_to_sequences(
    train_df["malayalam"].astype(str)
)
X_val = ml_tokenizer.texts_to_sequences(
    val_df["malayalam"].astype(str)
)
X_test = ml_tokenizer.texts_to_sequences(
    test_df["malayalam"].astype(str)
)

Y_train = en_tokenizer.texts_to_sequences(
    train_df["english"].astype(str)
)
Y_val = en_tokenizer.texts_to_sequences(
    val_df["english"].astype(str)
)
Y_test = en_tokenizer.texts_to_sequences(
    test_df["english"].astype(str)
)

print("Malayalam OOV rate:",
      round(oov_rate(X_train), 2), "%")
print("English OOV rate:",
      round(oov_rate(Y_train), 2), "%")


Malayalam OOV rate: 11.36 %
English OOV rate: 1.44 %


In [ ]:

import numpy as np

ml_lengths = [len(s) for s in X_train]
en_lengths = [len(s) for s in Y_train]

for name, lengths in [
    ("Malayalam", ml_lengths),
    ("English", en_lengths)
]:
    print(f"\n{name} sentence lengths")
    print("Median:", int(np.percentile(lengths, 50)))
    print("90th percentile:", int(np.percentile(lengths, 90)))
    print("95th percentile:", int(np.percentile(lengths, 95)))
    print("Maximum:", max(lengths))



Malayalam sentence lengths
Median: 10
90th percentile: 21
95th percentile: 26
Maximum: 72

English sentence lengths
Median: 15
90th percentile: 31
95th percentile: 38
Maximum: 75


In [ ]:
MAX_ML_LEN = 32

truncated_ml = sum(
    len(s) > MAX_ML_LEN for s in X_train
)

print("Malayalam sequence length:", MAX_ML_LEN)
print("Training sentences longer than limit:", truncated_ml)
print(
    "Percentage truncated:",
    round(100 * truncated_ml / len(X_train), 2),
    "%"
)

Malayalam sequence length: 32
Training sentences longer than limit: 1665
Percentage truncated: 1.44 %


In [ ]:
en_lengths = [len(s) for s in Y_train]

print("English median:", int(__import__('numpy').percentile(en_lengths, 50)))
print("English 90th percentile:", int(__import__('numpy').percentile(en_lengths, 90)))
print("English 95th percentile:", int(__import__('numpy').percentile(en_lengths, 95)))
print("English maximum:", max(en_lengths))

English median: 15
English 90th percentile: 31
English 95th percentile: 38
English maximum: 75


In [ ]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_ML_LEN = 32
MAX_EN_LEN = 42

# Pad Malayalam input sequences
X_train_pad = pad_sequences(
    X_train, maxlen=MAX_ML_LEN,
    padding="post", truncating="post"
)
X_val_pad = pad_sequences(
    X_val, maxlen=MAX_ML_LEN,
    padding="post", truncating="post"
)
X_test_pad = pad_sequences(
    X_test, maxlen=MAX_ML_LEN,
    padding="post", truncating="post"
)

# Pad English target sequences (initial form)
Y_train_pad = pad_sequences(
    Y_train, maxlen=MAX_EN_LEN,
    padding="post", truncating="post"
)
Y_val_pad = pad_sequences(
    Y_val, maxlen=MAX_EN_LEN,
    padding="post", truncating="post"
)
Y_test_pad = pad_sequences(
    Y_test, maxlen=MAX_EN_LEN,
    padding="post", truncating="post"
)

print("X_train:", X_train_pad.shape)
print("Y_train:", Y_train_pad.shape)
print("X_val:", X_val_pad.shape)
print("Y_val:", Y_val_pad.shape)
print("X_test:", X_test_pad.shape)
print("Y_test:", Y_test_pad.shape)


X_train: (116016, 32)
Y_train: (116016, 42)
X_val: (14502, 32)
Y_val: (14502, 42)
X_test: (14503, 32)
Y_test: (14503, 42)


In [ ]:
import pickle

with open("/content/ml_tokenizer.pkl", "wb") as f:
    pickle.dump(ml_tokenizer, f)

with open("/content/en_tokenizer.pkl", "wb") as f:
    pickle.dump(en_tokenizer, f)

print("Tokenizers saved successfully!")

Tokenizers saved successfully!


In [ ]:
from google.colab import files

files.download("/content/ml_tokenizer.pkl")
files.download("/content/en_tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np

np.savez_compressed(
    "/content/preprocessed_data.npz",
    X_train=X_train_pad,
    Y_train=Y_train_pad,
    X_val=X_val_pad,
    Y_val=Y_val_pad,
    X_test=X_test_pad,
    Y_test=Y_test_pad
)

print("Preprocessed arrays saved successfully!")

Preprocessed arrays saved successfully!


In [ ]:
from google.colab import files

files.download("/content/preprocessed_data.npz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
import zipfile

files_to_send = [
    "cleaned_dataset.csv",
    "train.csv",
    "validation.csv",
    "test.csv",
    "preprocessed_data.npz",
    "ml_tokenizer.pkl",
    "en_tokenizer.pkl"
]

zip_path = "/content/Group11_Preprocessed_Dataset.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in files_to_send:
        filepath = "/content/" + filename

        if os.path.exists(filepath):
            zipf.write(filepath, arcname=filename)
            print("Added:", filename)
        else:
            print("MISSING:", filename)

print("\nZIP file created:", os.path.exists(zip_path))


Added: cleaned_dataset.csv
Added: train.csv
Added: validation.csv
Added: test.csv
Added: preprocessed_data.npz
Added: ml_tokenizer.pkl
Added: en_tokenizer.pkl

ZIP file created: True


In [ ]:
from google.colab import files

files.download("/content/Group11_Preprocessed_Dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>